In [92]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [93]:
from pathlib import Path
from datetime import date, timedelta
import math
import random

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn
from tqdm.auto import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"
EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEST_START = date(2020, 9, 16)

K = 12
TOP_N = 100
HISTORY_DAYS = 56
DECAY_HALF_LIFE = 3
VISUAL_HALF_LIFE = 7
SEED = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)

Device: cuda


In [94]:
weights = np.load(PROCESSED_PATH / "final_hybrid_weights.npz")

HISTORY_WEIGHT = float(weights["history_weight"])
SASREC_WEIGHT = float(weights["sasrec_weight"])
DECAY_WEIGHT = float(weights["decay_weight"])
VISUAL_WEIGHT = float(weights["visual_weight"])

transactions = pl.scan_parquet(PROCESSED_PATH / "transactions_mapped.parquet")
test_ground_truth = pl.read_parquet(PROCESSED_PATH / "test_ground_truth.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")

NUM_ITEMS = article_mapping.height + 1

print("Weights:")
print("History:", HISTORY_WEIGHT)
print("SASRec:", SASREC_WEIGHT)
print("DecayPop:", DECAY_WEIGHT)
print("CLIP:", VISUAL_WEIGHT)

print("Test users:", test_ground_truth.height)
print("Items:", NUM_ITEMS - 1)

Weights:
History: 1.0
SASRec: 0.11999999731779099
DecayPop: 0.07999999821186066
CLIP: 0.07000000029802322
Test users: 68984
Items: 105542


In [95]:
assert test_ground_truth.height == 68984
assert NUM_ITEMS == 105543
assert HISTORY_WEIGHT == 1.0
assert 0 < SASREC_WEIGHT < 1
assert 0 < DECAY_WEIGHT < 1
assert 0 < VISUAL_WEIGHT < 1

print("Tests passed")

Tests passed


In [96]:
test_users = test_ground_truth.select("customer_idx")

history_56 = (
    transactions
    .filter(
        (pl.col("t_dat") < TEST_START)
        & (pl.col("t_dat") >= TEST_START - timedelta(days=HISTORY_DAYS))
    )
    .join(test_users.lazy(), on="customer_idx", how="semi")
    .sort(
        ["customer_idx", "t_dat"],
        descending=[False, True]
    )
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx")
        .unique(maintain_order=True)
        .head(TOP_N)
        .alias("history")
    )
    .collect()
)

evaluation_data = test_ground_truth.join(
    history_56,
    on="customer_idx",
    how="left"
)

actuals = evaluation_data["actual"].to_list()
history_lists = [
    history if history is not None else []
    for history in evaluation_data["history"].to_list()
]

print("Test users:", len(actuals))
print(
    "Users with 56d history:",
    sum(bool(history) for history in history_lists)
)

Test users: 68984
Users with 56d history: 42537


In [97]:
decay_top100 = (
    transactions
    .filter(pl.col("t_dat") < TEST_START)
    .with_columns(
        (TEST_START - pl.col("t_dat"))
        .dt.total_days()
        .cast(pl.Float32)
        .alias("age_days")
    )
    .with_columns(
        (
            -math.log(2)
            * pl.col("age_days")
            / DECAY_HALF_LIFE
        )
        .exp()
        .alias("score")
    )
    .group_by("article_idx")
    .agg(pl.col("score").sum())
    .sort("score", descending=True)
    .head(TOP_N)
    .collect()["article_idx"]
    .to_list()
)

decay_top100[:12]

[103109,
 104554,
 95218,
 104073,
 67523,
 3092,
 104046,
 103797,
 53893,
 104528,
 103794,
 82629]

In [98]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        max_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.item_embedding = nn.Embedding(
            num_items,
            hidden_dim,
            padding_idx=0
        )

        self.position_embedding = nn.Embedding(
            max_len,
            hidden_dim
        )

        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, input_items):
        seq_len = input_items.size(1)

        positions = torch.arange(
            seq_len,
            device=input_items.device
        ).unsqueeze(0)

        x = self.item_embedding(input_items)
        x = x * math.sqrt(self.hidden_dim)
        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_items.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        x = self.norm(x)

        return x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

In [99]:
checkpoint_candidates = []

for path in list(CHECKPOINTS_PATH.glob("*.pt")) + list(CHECKPOINTS_PATH.glob("*.pth")):
    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        state = checkpoint.get("model_state_dict", {})

        if (
            "item_embedding.weight" in state
            and "metadata_gate_logit" not in state
            and "max_len" in checkpoint
            and "hidden_dim" in checkpoint
        ):
            map12 = checkpoint.get(
                "metrics",
                {}
            ).get("MAP@12", -1)

            checkpoint_candidates.append(
                (map12, path, checkpoint)
            )
    except Exception:
        pass

assert checkpoint_candidates, "SASRec checkpoint not found"

checkpoint_candidates.sort(
    key=lambda x: x[0],
    reverse=True
)

_, SASREC_CHECKPOINT_PATH, checkpoint = checkpoint_candidates[0]

print("Checkpoint:", SASREC_CHECKPOINT_PATH.name)
print("Epoch:", checkpoint.get("epoch"))
print("MAP@12:", checkpoint.get("metrics", {}).get("MAP@12"))

Checkpoint: sasrec_best.pt
Epoch: 8
MAP@12: 0.014934833159714998


In [100]:
MAX_LEN = checkpoint["max_len"]

sequences = (
    transactions
    .filter(pl.col("t_dat") < TEST_START)
    .join(test_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").alias("sequence"))
    .collect()
)

sequence_dict = dict(
    zip(
        sequences["customer_idx"].to_list(),
        sequences["sequence"].to_list()
    )
)

customer_indices = test_ground_truth["customer_idx"].to_list()

sequence_array = np.zeros(
    (len(customer_indices), MAX_LEN),
    dtype=np.int32
)

has_sequence = np.zeros(
    len(customer_indices),
    dtype=bool
)

for i, customer_idx in enumerate(customer_indices):
    sequence = sequence_dict.get(customer_idx)

    if not sequence:
        continue

    sequence = sequence[-MAX_LEN:]

    sequence_array[
        i,
        -len(sequence):
    ] = sequence

    has_sequence[i] = True

print("Shape:", sequence_array.shape)
print("Users with sequence:", has_sequence.sum())

Shape: (68984, 64)
Users with sequence: 63412


In [101]:
model = SASRec(
    num_items=NUM_ITEMS,
    max_len=checkpoint["max_len"],
    hidden_dim=checkpoint["hidden_dim"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"]
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded SASRec")

Loaded SASRec


In [102]:
sasrec_top100 = np.zeros(
    (len(actuals), TOP_N),
    dtype=np.int32
)

valid_user_indices = np.flatnonzero(has_sequence)

BATCH_SIZE = 256

with torch.no_grad():
    for start in tqdm(
        range(0, len(valid_user_indices), BATCH_SIZE),
        desc="SASRec test retrieval"
    ):
        indices = valid_user_indices[
            start:start + BATCH_SIZE
        ]

        batch = torch.from_numpy(
            sequence_array[indices]
        ).long().to(DEVICE)

        with torch.amp.autocast(
            "cuda",
            enabled=DEVICE.type == "cuda",
            dtype=torch.float16
        ):
            hidden = model(batch)
            user_embeddings = hidden[:, -1]

            scores = (
                user_embeddings
                @ model.item_embedding.weight.T
            )

            scores[:, 0] = -torch.inf

            top_items = torch.topk(
                scores,
                TOP_N,
                dim=1
            ).indices

        sasrec_top100[indices] = (
            top_items.cpu().numpy().astype(np.int32)
        )

print("Shape:", sasrec_top100.shape)

SASRec test retrieval:   0%|          | 0/248 [00:00<?, ?it/s]

Shape: (68984, 100)


In [104]:
visual_candidates = []

for path in EMBEDDINGS_PATH.rglob("*.npy"):
    try:
        array = np.load(path, mmap_mode="r")

        if array.shape == (NUM_ITEMS, 512):
            visual_candidates.append(path)
    except Exception:
        pass

assert visual_candidates, "CLIP embeddings not found"

VISUAL_EMBEDDINGS_PATH = visual_candidates[0]

visual_embeddings = np.load(
    VISUAL_EMBEDDINGS_PATH,
    mmap_mode="r"
)

assert visual_embeddings.shape == (NUM_ITEMS, 512)

print("Path:", VISUAL_EMBEDDINGS_PATH)
print("Shape:", visual_embeddings.shape)
print("Dtype:", visual_embeddings.dtype)

Path: /content/drive/MyDrive/multimodal-fashion-recsys/embeddings/clip_vit_b32_embeddings.npy
Shape: (105543, 512)
Dtype: float16


In [105]:
visual_history = (
    transactions
    .filter(
        (pl.col("t_dat") < TEST_START)
        & (pl.col("t_dat") >= TEST_START - timedelta(days=HISTORY_DAYS))
    )
    .join(test_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx").alias("history"),
        pl.col("t_dat").alias("dates")
    )
    .collect()
)

visual_test_data = (
    test_ground_truth
    .select("customer_idx")
    .join(
        visual_history,
        on="customer_idx",
        how="left"
    )
)

visual_test_data.head()

customer_idx,history,dates
u32,list[u32],list[date]
1135845,[50856],[2020-08-03]
1065724,"[99199, 70568, … 98298]","[2020-08-05, 2020-08-05, … 2020-08-05]"
169340,"[70998, 42938, … 98091]","[2020-08-14, 2020-08-14, … 2020-09-09]"
557039,"[72478, 96901, … 103076]","[2020-07-29, 2020-07-29, … 2020-09-11]"
1110100,null,null


In [106]:
visual_profiles = np.zeros(
    (len(actuals), 512),
    dtype=np.float32
)

has_visual_profile = np.zeros(
    len(actuals),
    dtype=bool
)

histories = visual_test_data["history"].to_list()
dates = visual_test_data["dates"].to_list()

for i, (history, history_dates) in enumerate(
    tqdm(
        zip(histories, dates),
        total=len(histories),
        desc="Building visual profiles"
    )
):
    if not history:
        continue

    item_ids = np.asarray(
        history,
        dtype=np.int32
    )

    item_embeddings = visual_embeddings[
        item_ids
    ].astype(np.float32)

    valid = (
        np.linalg.norm(
            item_embeddings,
            axis=1
        ) > 0
    )

    if not valid.any():
        continue

    ages = np.asarray(
        [
            (TEST_START - d).days
            for d in history_dates
        ],
        dtype=np.float32
    )[valid]

    recency_weights = (
        0.5 ** (
            ages / VISUAL_HALF_LIFE
        )
    )

    profile = (
        item_embeddings[valid]
        * recency_weights[:, None]
    ).sum(axis=0)

    profile /= recency_weights.sum()

    norm = np.linalg.norm(profile)

    if norm > 0:
        visual_profiles[i] = profile / norm
        has_visual_profile[i] = True

print(
    "Users with visual profile:",
    has_visual_profile.sum()
)

Building visual profiles:   0%|          | 0/68984 [00:00<?, ?it/s]

Users with visual profile: 42533


In [107]:
visual_top100 = np.zeros(
    (len(actuals), TOP_N),
    dtype=np.int32
)

embedding_norms = np.linalg.norm(
    visual_embeddings.astype(np.float32),
    axis=1
)

valid_items = embedding_norms > 0
valid_items[0] = False

item_embeddings_gpu = torch.from_numpy(
    np.asarray(
        visual_embeddings,
        dtype=np.float32
    )
).to(
    DEVICE,
    dtype=torch.float16
)

valid_items_gpu = torch.from_numpy(
    valid_items
).to(DEVICE)

visual_user_indices = np.flatnonzero(
    has_visual_profile
)

BATCH_SIZE = 256

with torch.no_grad():
    for start in tqdm(
        range(0, len(visual_user_indices), BATCH_SIZE),
        desc="CLIP test retrieval"
    ):
        indices = visual_user_indices[
            start:start + BATCH_SIZE
        ]

        profiles = torch.from_numpy(
            visual_profiles[indices]
        ).to(
            DEVICE,
            dtype=torch.float16
        )

        scores = (
            profiles
            @ item_embeddings_gpu.T
        )

        scores[:, ~valid_items_gpu] = -torch.inf

        top_items = torch.topk(
            scores,
            TOP_N,
            dim=1
        ).indices

        visual_top100[indices] = (
            top_items.cpu().numpy().astype(np.int32)
        )

print("Shape:", visual_top100.shape)

CLIP test retrieval:   0%|          | 0/167 [00:00<?, ?it/s]

Shape: (68984, 100)


In [108]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    used = set()

    for i, item in enumerate(predicted[:k]):
        if item in actual and item not in used:
            hits += 1
            score += hits / (i + 1)
            used.add(item)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    hits = len(
        actual.intersection(predicted[:k])
    )

    return hits / min(len(actual), k)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(
        1 / math.log2(i + 2)
        for i, item in enumerate(predicted[:k])
        if item in actual
    )

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(min(len(actual), k))
    )

    return dcg / idcg

In [109]:
def fuse_rankings(
    history,
    sasrec,
    decay,
    visual,
    history_weight,
    sasrec_weight,
    decay_weight,
    visual_weight,
    top_n
):
    scores = {}

    rankings = [
        (history, history_weight),
        (sasrec, sasrec_weight),
        (decay, decay_weight),
        (visual, visual_weight)
    ]

    for ranking, weight in rankings:
        for rank, item in enumerate(ranking):
            item = int(item)

            if item <= 0:
                continue

            scores[item] = (
                scores.get(item, 0.0)
                + weight / (rank + 1)
            )

    return [
        item
        for item, _ in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]
    ]

In [110]:
test_predictions = [
    fuse_rankings(
        history=history_lists[i],
        sasrec=sasrec_top100[i],
        decay=decay_top100,
        visual=visual_top100[i],
        history_weight=HISTORY_WEIGHT,
        sasrec_weight=SASREC_WEIGHT,
        decay_weight=DECAY_WEIGHT,
        visual_weight=VISUAL_WEIGHT,
        top_n=TOP_N
    )
    for i in tqdm(
        range(len(actuals)),
        desc="Final test hybrid"
    )
]

Final test hybrid:   0%|          | 0/68984 [00:00<?, ?it/s]

In [111]:
test_metrics = {
    "MAP@12": sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, test_predictions)
    ) / len(actuals),

    "Recall@12": sum(
        recall_at_k(a, p, K)
        for a, p in zip(actuals, test_predictions)
    ) / len(actuals),

    "NDCG@12": sum(
        ndcg_at_k(a, p, K)
        for a, p in zip(actuals, test_predictions)
    ) / len(actuals),

    "Coverage": len({
        item
        for prediction in test_predictions
        for item in prediction[:K]
    }) / (NUM_ITEMS - 1)
}

test_metrics

{'MAP@12': 0.026336378529433845,
 'Recall@12': 0.055939966692822425,
 'NDCG@12': 0.03848529230984866,
 'Coverage': 0.328229520001516}

In [112]:
validation_metrics = {
    "MAP@12": float(weights["validation_map12"]),
    "Recall@12": float(weights["validation_recall12"]),
    "NDCG@12": float(weights["validation_ndcg12"]),
    "Coverage": float(weights["validation_coverage"])
}

final_evaluation = pl.DataFrame([
    {
        "split": "Validation",
        **validation_metrics
    },
    {
        "split": "Test",
        **test_metrics
    }
])

final_evaluation

split,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Validation""",0.027367,0.056281,0.039973,0.243676
"""Test""",0.026336,0.05594,0.038485,0.32823
